<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/Sentence_Transformer_Chunk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q sentence-transformers faiss-cpu openai

# Import libraries
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

# Load API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("WEEK2_KEY")

# Initialize OpenAI client
client = OpenAI()

# Dataset used for retrieval
dataset = """
Artificial intelligence (AI) refers to the simulation of human intelligence in machines.
AI systems are capable of learning, reasoning, problem-solving, perception, and language understanding.
Machine learning is a subset of AI that focuses on learning from data.
Deep learning uses neural networks with multiple layers.
Large language models are trained on vast amounts of text data.
These models can perform tasks like translation, summarization, and question answering.
Embeddings are numerical representations of text or data.
Embeddings capture semantic meaning in vector space.
Vector databases store embeddings efficiently.
Similarity search retrieves vectors closest to a query vector.
FAISS is a library developed by Facebook AI Research for efficient similarity search.
FAISS supports clustering, indexing, and fast nearest neighbor search.
Retrieval Augmented Generation (RAG) combines retrieval with text generation.
RAG improves factual accuracy of language models.
In RAG systems, embeddings are retrieved before generating responses.
"""

# Split dataset into chunks
def chunk_text(text):
    return [line.strip() for line in text.split("\n") if line.strip()]

chunks = chunk_text(dataset)

# Load sentence-transformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for chunks
embeddings = embedding_model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

# Store embeddings in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Retrieve top-k relevant chunks
def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

# Generate answer using retrieved context
def rag_chat(question, top_k=3):
    retrieved_chunks = retrieve(question, top_k)
    context = "\n".join(retrieved_chunks)

    prompt = f"""
Use ONLY the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=0
    )

    return response.output_text

# Interactive chat loop
print("RAG Chatbot is ready ✅")

while True:
    user_question = input("You: ")
    if user_question.lower() in ["exit", "quit", "stop"]:
        break
    print("Bot:", rag_chat(user_question))